# backprop-pop-outgrad-loop — worked example 1: Reverse-pass driver over a straight-line chain (mul then add)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

The backprop driver walks a **reverse-topological** list of graph nodes. For each node it **pops** its accumulated outgrad from a `grads` dict (so the node is done after), then either writes to `.grad` (leaf) or dispatches each parent's `back_fn` and **accumulates** the result into `grads[parent]`. Popping (not peeking) guarantees each node is finalised exactly once; accumulating with `+` is what makes shared parents in a DAG correct.

## Worked solution

We build a tiny graph for `y = (a * b) + c` where `a`, `b`, `c` are leaves. The reverse-topo order is `[y, prod, a, b, c]` (end first, leaves last).

**Step 1 — seed the accumulator.** `grads = {id(y): end_grad}` with `end_grad = ones_like(y)`. This is `dL/dy` when `L = sum(y)`, the conventional starting seed.

**Step 2 — pop `y`.** `y` is a non-leaf (it has a recipe for `add`). We pop its grad, then for each parent (`prod` at arg 0, `c` at arg 1) we call the add back_fn. Add's back_fn just passes the upstream grad through unchanged, so `grads[prod] += g` and `grads[c] += g`. We pop rather than peek so a later stray read of `grads[id(y)]` would `KeyError` — a guard against double-processing.

**Step 3 — pop `prod`.** Non-leaf (`mul`). Its parents are `a` (arg 0) and `b` (arg 1). Mul's back_fn w.r.t. arg 0 is `grad_out * b`, w.r.t. arg 1 is `grad_out * a`. We accumulate each into the matching leaf's slot.

**Step 4 — pop `a`, `b`, `c`.** Each is a leaf (`recipe is None`), so we write into `.grad`, accumulating with `+` if `.grad` was already set.

**Why it works.** Processing in reverse-topo order guarantees every node's full incoming grad has been accumulated before it is popped — all of its consumers come earlier in the list. Verifying against analytic gradients: `dy/da = b`, `dy/db = a`, `dy/dc = 1`.

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x
def _add_back0(go, out, x, y):  return go
def _add_back1(go, out, x, y):  return go

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

# Build graph for y = (a*b) + c
t.manual_seed(0)
a = MiniTensor(t.randn(3)); b = MiniTensor(t.randn(3)); c = MiniTensor(t.randn(3))
prod_arr = a.array * b.array
prod = MiniTensor(prod_arr, Recipe('mul', (a.array, b.array), {}, {0: a, 1: b}))
y_arr = prod_arr + c.array
y = MiniTensor(y_arr, Recipe('add', (prod_arr, c.array), {}, {0: prod, 1: c}))

back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1,
              ('add', 0): _add_back0, ('add', 1): _add_back1}
sorted_graph = [y, prod, a, b, c]
backprop(y, t.ones_like(y.array), sorted_graph, back_funcs)
print('a.grad == b :', t.allclose(a.grad, b.array))
print('b.grad == a :', t.allclose(b.grad, a.array))
print('c.grad == ones:', t.allclose(c.grad, t.ones(3)))